# 03 · Model Training and Comparison (8 models)

Train and compare candidate classifiers.

Protocol
  * Fit/CV: 80 TRAIN trips, 5-fold GroupKFold (groups = trip, so no trip is split across folds).
  * Comparison: 10 STUDENT-TEST trips (never used for fitting).
  * Only GPS points with a reliable reference label (<= 50 m from aligned route arc) are used.
Outputs
  results/cv_folds.csv, results/model_comparison.csv, results/model_configs.json,
  results/history_<model>.csv (epoch/iteration-wise histories where the algorithm has one),
  results/test_scores_<model>.pkl (student-test candidate probabilities)

In [1]:
import sys, time, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent          # run this notebook from the notebooks/ folder
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from pathsnap import config as C
from pathsnap.models import MODEL_SPECS, make_model, load_split, subsample_points, X_of
from pathsnap.metrics import candidate_metrics, top1_predictions, top1_accuracy, route_continuity
from pathsnap.data_io import load_trip

In [2]:
PREDS = C.RESULTS / "preds"; PREDS.mkdir(parents=True, exist_ok=True)

In [3]:
def fit(name, model, X, y, rng):
    cap = MODEL_SPECS[name][2]
    if cap and len(X) > cap:
        i = rng.choice(len(X), cap, replace=False); X, y = X[i], y[i]
    t0 = time.time(); model.fit(X, y); return time.time() - t0

In [4]:
def score_split(cands, pts, p):
    c = cands.assign(p=p)
    pred = top1_predictions(c)
    return c, pred, top1_accuracy(pts, pred)

In [5]:
def continuity_for(cands_all_pts_scored, split_dir):
    """Route continuity of classifier-only Top-1 over ALL points of each trip."""
    vals = []
    pred = top1_predictions(cands_all_pts_scored)
    for tid, g in pred.groupby("trip"):
        trip = load_trip(split_dir / tid, with_route=False)
        arr = np.full(len(trip.gps_xy), np.nan); arr[g.pt.to_numpy()] = g.pred_arc.to_numpy()
        vals.append(route_continuity(arr, trip.arcs))
    return float(np.nanmean(vals))

In [6]:
only = list(MODEL_SPECS)   # or e.g. ["XGBoost", "LightGBM"] to re-run a subset
rng = np.random.default_rng(C.SEED)
tr_c, tr_p = load_split("train")
tr_c, tr_p = subsample_points(tr_c, tr_p)
te_c, te_p = load_split("student_test")
te_all = pd.read_pickle(C.PROCESSED / "student_test_cands.pkl")  # all points, for continuity
Xtr, ytr, gtr = X_of(tr_c), tr_c.label.to_numpy(), tr_c.trip.to_numpy()
print(f"train rows={len(Xtr):,} points={len(tr_p):,} trips={tr_c.trip.nunique()} | "
      f"test rows={len(te_c):,} points={len(te_p):,}", flush=True)

folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
cv_rows, comp_rows = [], []
for name in only:
    family = MODEL_SPECS[name][0]
    for k, (a, b) in enumerate(folds):
        m = make_model(name)
        tfit = fit(name, m, Xtr[a], ytr[a], rng)
        pb = m.predict_proba(Xtr[b])[:, 1]
        cm = candidate_metrics(ytr[b], pb)
        vc = tr_c.iloc[b]; vp = tr_p[tr_p.trip.isin(set(vc.trip))]
        _, _, acc = score_split(vc, vp, pb)
        cv_rows.append(dict(model=name, fold=k, top1=acc, fit_s=tfit, **cm))
        print(f"  {name} fold{k}: f1_macro={cm['f1_macro']:.4f} top1={acc:.4f} logloss={cm['log_loss']:.4f} fit={tfit:.1f}s", flush=True)
    cvd = pd.DataFrame([r for r in cv_rows if r["model"] == name])

    # refit on all training data, evaluate on student test
    m = make_model(name)
    tfit = fit(name, m, Xtr, ytr, rng)
    tr_pred = m.predict_proba(Xtr)[:, 1]
    tr_cm = candidate_metrics(ytr, tr_pred)
    _, _, tr_acc = score_split(tr_c, tr_p, tr_pred)
    t0 = time.time(); pte = m.predict_proba(X_of(te_c))[:, 1]; t_inf = time.time() - t0
    cm = candidate_metrics(te_c.label.to_numpy(), pte)
    _, _, acc = score_split(te_c, te_p, pte)
    cont = continuity_for(te_all.assign(p=m.predict_proba(X_of(te_all))[:, 1]), C.STUDENT_TEST_DIR)
    te_c[["trip", "pt", "arc", "label"]].assign(p=pte).to_pickle(PREDS / f"test_scores_{name}.pkl")

    # epoch-/iteration-wise histories (only for algorithms that actually have them)
    if name == "MLP":
        mlp = m[-1]
        pd.DataFrame({"epoch": np.arange(1, len(mlp.loss_curve_) + 1), "train_loss": mlp.loss_curve_,
                      "val_accuracy": mlp.validation_scores_}).to_csv(C.RESULTS / "history_MLP.csv", index=False)
    if name in ("LightGBM", "XGBoost"):
        a, b = folds[0]
        h = make_model(name)
        if name == "LightGBM":
            h.fit(Xtr[a], ytr[a], eval_set=[(Xtr[a], ytr[a]), (Xtr[b], ytr[b])], eval_metric="binary_logloss")
            r = list(h.evals_result_.values()); tl, vl = r[0]["binary_logloss"], r[1]["binary_logloss"]
        else:
            h.fit(Xtr[a], ytr[a], eval_set=[(Xtr[a], ytr[a]), (Xtr[b], ytr[b])], verbose=False)
            r = h.evals_result(); tl, vl = r["validation_0"]["logloss"], r["validation_1"]["logloss"]
        pd.DataFrame({"iteration": np.arange(1, len(tl) + 1), "train_logloss": tl, "val_logloss": vl}) \
            .to_csv(C.RESULTS / f"history_{name}.csv", index=False)

    comp_rows.append(dict(
        model=name, family=family, config=json.dumps(MODEL_SPECS[name][1]),
        cv_f1_macro_mean=cvd.f1_macro.mean(), cv_f1_macro_std=cvd.f1_macro.std(),
        cv_top1_mean=cvd.top1.mean(), cv_top1_std=cvd.top1.std(), cv_log_loss=cvd.log_loss.mean(),
        train_top1=tr_acc, train_f1_macro=tr_cm["f1_macro"], train_log_loss=tr_cm["log_loss"],
        test_top1=acc, test_f1_macro=cm["f1_macro"], test_precision_macro=cm["precision_macro"],
        test_recall_macro=cm["recall_macro"], test_log_loss=cm["log_loss"], test_accuracy=cm["accuracy"],
        test_route_continuity=cont, fit_time_s=tfit,
        inference_ms_per_1k_rows=1000 * t_inf / len(te_c) * 1000))
    print(pd.Series(comp_rows[-1]).to_string(), flush=True)

cv = pd.DataFrame(cv_rows); comp = pd.DataFrame(comp_rows)

train rows=760,023 points=97,238 trips=80 | test rows=147,002 points=19,133


  LogisticRegression fold0: f1_macro=0.9060 top1=0.8612 logloss=0.1123 fit=1.9s


  LogisticRegression fold1: f1_macro=0.8941 top1=0.8461 logloss=0.1339 fit=1.7s


  LogisticRegression fold2: f1_macro=0.9164 top1=0.9026 logloss=0.1034 fit=1.8s


  LogisticRegression fold3: f1_macro=0.9035 top1=0.8720 logloss=0.1269 fit=1.8s


  LogisticRegression fold4: f1_macro=0.9190 top1=0.8998 logloss=0.1053 fit=1.7s


... (40 outputs left out)

  MLP fold2: f1_macro=0.9439 top1=0.9140 logloss=0.0672 fit=20.9s


  MLP fold3: f1_macro=0.9319 top1=0.8896 logloss=0.0853 fit=30.4s


  MLP fold4: f1_macro=0.9402 top1=0.9114 logloss=0.0785 fit=20.9s


model                                                                     MLP
family                                                         Neural network
config                      {"hidden_layer_sizes": [64, 32], "alpha": 0.00...
cv_f1_macro_mean                                                     0.933382
cv_f1_macro_std                                                      0.008513
cv_top1_mean                                                         0.889583
cv_top1_std                                                          0.023179
cv_log_loss                                                          0.080583
train_top1                                                           0.905284
train_f1_macro                                                       0.943054
train_log_loss                                                       0.066993
test_top1                                                            0.903413
test_f1_macro                                                   

### merge with any earlier runs of other models

In [7]:
for fname, df in [("cv_folds.csv", cv), ("model_comparison.csv", comp)]:
    f = C.RESULTS / fname
    if f.exists() and only != list(MODEL_SPECS):
        old = pd.read_csv(f); df = pd.concat([old[~old.model.isin(only)], df])
    df.to_csv(f, index=False)
json.dump({k: {"family": v[0], "params": v[1], "max_train_rows": v[2]} for k, v in MODEL_SPECS.items()},
          open(C.RESULTS / "model_configs.json", "w"), indent=2)
print("done")

done


In [8]:
pd.read_csv(C.RESULTS / "model_comparison.csv").round(4)

,model,family,config,cv_f1_macro_mean,cv_f1_macro_std,cv_top1_mean,cv_top1_std,cv_log_loss,train_top1,train_f1_macro,train_log_loss,test_top1,test_f1_macro,test_precision_macro,test_recall_macro,test_log_loss,test_accuracy,test_route_continuity,fit_time_s,inference_ms_per_1k_rows
0,LogisticRegression,Linear / probabilistic,"{""C"": 1.0, ""max_iter"": 500}",0.9078,0.0101,0.8764,0.0245,0.1163,0.8768,0.9087,0.1151,0.8716,0.9094,0.9107,0.9081,0.1106,0.9591,0.9213,2.1862,0.0863
1,GaussianNB,Linear / probabilistic,"{""var_smoothing"": 1e-09}",0.8378,0.0100,0.8598,0.0268,0.5448,0.8605,0.8379,0.5406,0.8479,0.8328,0.7924,0.9031,0.6025,0.9093,0.9167,0.1595,0.1637
2,KNN,Instance-based,"{""n_neighbors"": 25, ""weights"": ""distance""}",0.9160,0.0103,0.8707,0.0240,0.1368,0.8985,0.9297,0.1014,0.8751,0.9183,0.9311,0.9065,0.1269,0.9641,0.9243,0.0265,144.8919
3,DecisionTree,Tree-based,"{""max_depth"": 10, ""min_samples_leaf"": 20}",0.9313,0.0075,0.8844,0.0219,0.0895,0.9042,0.9434,0.0683,0.8903,0.9330,0.9371,0.9290,0.0799,0.9700,0.9313,11.5744,0.0859
4,RandomForest,Ensemble (bagging),"{""n_estimators"": 100, ""max_depth"": 16, ""min_sa...",0.9353,0.0086,0.8906,0.0237,0.0746,0.9154,0.9501,0.0564,0.8963,0.9356,0.9486,0.9237,0.0724,0.9717,0.9349,48.4678,3.3880
5,LightGBM,Boosting,"{""n_estimators"": 300, ""learning_rate"": 0.05, ""...",0.9389,0.0086,0.8957,0.0213,0.0716,0.9207,0.9526,0.0549,0.9052,0.9406,0.9456,0.9356,0.0685,0.9734,0.9404,9.4671,5.9172
6,XGBoost,Boosting,"{""n_estimators"": 300, ""learning_rate"": 0.05, ""...",0.9395,0.0086,0.8972,0.0211,0.0708,0.9210,0.9537,0.0546,0.9072,0.9403,0.9466,0.9342,0.0678,0.9733,0.9404,10.7542,2.0551
7,MLP,Neural network,"{""hidden_layer_sizes"": [64, 32], ""alpha"": 0.00...",0.9334,0.0085,0.8896,0.0232,0.0806,0.9053,0.9431,0.0670,0.9034,0.9363,0.9483,0.9252,0.0738,0.9719,0.9409,26.5642,0.3510
